In [1]:
# SETUP: Installing and importing packages
!pip install -q google-genai

from google import genai
from google.colab import userdata
import json
import re

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

In [2]:
# SETUP: Using the same extract_json as the main pipeline
def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        return json.loads(match.group())
    else:
        raise ValueError("No JSON found in response")

# SETUP: Retry wrapper — handles 503 overload errors gracefully
import time

def generate_with_retry(client, prompt, retries=4, wait=30):
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model='gemini-pro-latest',
                contents=prompt
            )
            return response
        except Exception as e:
            if attempt < retries - 1:
                print(f"  Error: {e}. Waiting {wait}s before retry {attempt + 2}/{retries}...")
                time.sleep(wait)
            else:
                raise

# **Step 1:** Creating Personas

In [3]:
jordan = {
    "name": "Jordan",
    "grade": "7th",
    "interests": ["gaming", "drawing", "YouTube"],
    "self_reported_comfort": "I usually just Google it or ask my older sister",
    "inferred_comfort": "beginner",
    "goals": "learn how to make and edit my own YouTube videos"
}

sarah = {
    "name": "Sarah",
    "grade": "10th",
    "interests": ["coding", "journalism", "podcasting"],
    "self_reported_comfort": "Pretty comfortable — I build my own websites and edit audio",
    "inferred_comfort": "advanced",
    "goals": "understand how AI-generated content affects what I read online"
}

marcus = {
    "name": "Marcus",
    "grade": "8th",
    "interests": ["basketball", "music production", "TikTok"],
    "self_reported_comfort": "I'm pretty good with tech, I use it all the time",
    "inferred_comfort": "beginner",
    "goals": "learn how to tell if something I see online is real or fake"
}

personas = {"Jordan": jordan, "Sarah": sarah, "Marcus": marcus}
print("Personas loaded:", list(personas.keys()))

Personas loaded: ['Jordan', 'Sarah', 'Marcus']


# **Step 2:** Combining Stages #2-3 into a function that personas can be run on

In [4]:
# STAGE #2

def run_stage2(student_profile):

    # --- Generate quiz ---
    quiz_prompt = f"""
You are a digital literacy assessment tool designing a short quiz for a specific student.
Your goal is to accurately assess their real skill level — not trick them, but genuinely
surface what they know and don't know across four areas of digital literacy.

Student profile:
{json.dumps(student_profile, indent=2)}

Generate exactly 6 multiple choice questions following these rules:

DISTRIBUTION (strictly enforce this):
- 2 questions on online safety
- 2 questions on media literacy
- 1 question on responsible AI use
- 1 question on file and device management

QUESTION QUALITY:
- Each question must have one clearly correct answer and three plausible wrong answers
- Difficulty should match inferred_comfort: "{student_profile['inferred_comfort']}"
  NOT self_reported_comfort
- At least 2 questions should use a scenario connected to the student's interests:
  {json.dumps(student_profile['interests'])}

AREA VALUES must be exactly one of:
  "online_safety" | "media_literacy" | "responsible_ai_use" | "file_device_management"

Return ONLY a JSON object in this exact format, no extra text or markdown:
{{
    "questions": [
        {{"id": "Q1", "area": "", "question": "", "options": ["A. ", "B. ", "C. ", "D. "], "correct_answer": "A"}},
        {{"id": "Q2", "area": "", "question": "", "options": ["A. ", "B. ", "C. ", "D. "], "correct_answer": "A"}},
        {{"id": "Q3", "area": "", "question": "", "options": ["A. ", "B. ", "C. ", "D. "], "correct_answer": "A"}},
        {{"id": "Q4", "area": "", "question": "", "options": ["A. ", "B. ", "C. ", "D. "], "correct_answer": "A"}},
        {{"id": "Q5", "area": "", "question": "", "options": ["A. ", "B. ", "C. ", "D. "], "correct_answer": "A"}},
        {{"id": "Q6", "area": "", "question": "", "options": ["A. ", "B. ", "C. ", "D. "], "correct_answer": "A"}}
    ]
}}
"""
    quiz_response = generate_with_retry(client, quiz_prompt)
    quiz_data = extract_json(quiz_response.text)

    # --- Simulate student answers ---
    # For evaluation purposes we simulate a realistic answer pattern based on inferred_comfort:
    # beginner gets ~33% right, intermediate ~60%, advanced ~85%
    import random
    random.seed(42)  # fixed seed for reproducibility

    accuracy_by_level = {"beginner": 0.33, "intermediate": 0.60, "advanced": 0.85}
    accuracy = accuracy_by_level.get(student_profile["inferred_comfort"], 0.5)

    student_answers = {}
    for q in quiz_data["questions"]:
        if random.random() < accuracy:
            student_answers[q["id"]] = q["correct_answer"]
        else:
            wrong = [opt[0] for opt in q["options"] if opt[0] != q["correct_answer"]]
            student_answers[q["id"]] = random.choice(wrong)

    # --- Score and build skill matrix ---
    skill_matrix = {
        "online_safety":          {"score": 0, "note": ""},
        "media_literacy":         {"score": 0, "note": ""},
        "responsible_ai_use":     {"score": 0, "note": ""},
        "file_device_management": {"score": 0, "note": ""}
    }
    area_totals = {area: {"correct": 0, "total": 0} for area in skill_matrix}

    for q in quiz_data["questions"]:
        area = q["area"]
        if area not in area_totals:
            continue
        area_totals[area]["total"] += 1
        if student_answers.get(q["id"]) == q["correct_answer"]:
            area_totals[area]["correct"] += 1

    def score_note(correct, total):
        ratio = correct / total if total > 0 else 0
        if ratio == 1.0:   return "strong — got everything right here"
        elif ratio >= 0.5: return "partial — some gaps worth addressing"
        else:              return "needs work — missed most questions in this area"

    for area, counts in area_totals.items():
        skill_matrix[area]["score"] = counts["correct"]
        skill_matrix[area]["note"]  = score_note(counts["correct"], counts["total"])

    total_correct   = sum(v["correct"] for v in area_totals.values())
    total_questions = sum(v["total"]   for v in area_totals.values())
    ratio = total_correct / total_questions if total_questions > 0 else 0

    if ratio >= 0.75:   overall_level = "advanced"
    elif ratio >= 0.4:  overall_level = "intermediate"
    else:               overall_level = "beginner"

    return quiz_data, student_answers, skill_matrix, overall_level

In [10]:
# STAGE #3

def run_stage3(student_profile, skill_matrix, overall_level):

    area_scores = {area: data["score"] for area, data in skill_matrix.items()}
    prioritized_areas = sorted(area_scores, key=lambda a: area_scores[a])

    lesson_prompt = f"""
You are designing a personalized digital literacy curriculum for a specific student.
Generate all 4 lessons in a single response — one for each area below, in the order listed.
Make each lesson genuinely engaging for this student — not a generic template.

Student profile: {json.dumps(student_profile, indent=2)}
Skill matrix: {json.dumps(skill_matrix, indent=2)}
Overall level: {overall_level}

Generate lessons in this priority order (weakest area first):
{json.dumps(prioritized_areas, indent=2)}

Requirements for each lesson:
- Calibrate difficulty to "{overall_level}"
- Open with a hook tied to their interests: {json.dumps(student_profile["interests"])}
- Activity must be concrete and completable in one sitting

Return ONLY a JSON object, no extra text or markdown:
{{
    "lessons": [
        {{
            "title": "",
            "area": "",
            "learning_objective": "Students will be able to...",
            "summary": "",
            "interest_hook": "",
            "activity": "",
            "estimated_duration": ""
        }},
        {{
            "title": "",
            "area": "",
            "learning_objective": "Students will be able to...",
            "summary": "",
            "interest_hook": "",
            "activity": "",
            "estimated_duration": ""
        }},
        {{
            "title": "",
            "area": "",
            "learning_objective": "Students will be able to...",
            "summary": "",
            "interest_hook": "",
            "activity": "",
            "estimated_duration": ""
        }},
        {{
            "title": "",
            "area": "",
            "learning_objective": "Students will be able to...",
            "summary": "",
            "interest_hook": "",
            "activity": "",
            "estimated_duration": ""
        }}
    ]
}}
"""
    response = generate_with_retry(client, lesson_prompt)
    result = extract_json(response.text)
    curriculum = result["lessons"]

    return prioritized_areas, curriculum

# **Step 3:** Running the three personas through the pipeline

In [6]:
# Running the three personas through the pipeline

results = {}

for name, profile in personas.items():
    print(f"Running pipeline for {name}...")
    quiz_data, student_answers, skill_matrix, overall_level = run_stage2(profile)
    prioritized_areas, curriculum = run_stage3(profile, skill_matrix, overall_level)

    results[name] = {
        "profile":           profile,
        "quiz_data":         quiz_data,
        "student_answers":   student_answers,
        "skill_matrix":      skill_matrix,
        "overall_level":     overall_level,
        "prioritized_areas": prioritized_areas,
        "curriculum":        curriculum
    }
    print(f"  Done. Overall level: {overall_level}, priority order: {prioritized_areas}\n")
    time.sleep(30)  # pause between personas to stay under rate limit

print("All personas complete.")

Running pipeline for Jordan...
  Done. Overall level: beginner, priority order: ['online_safety', 'responsible_ai_use', 'media_literacy', 'file_device_management']

Running pipeline for Sarah...
  Done. Overall level: advanced, priority order: ['responsible_ai_use', 'file_device_management', 'online_safety', 'media_literacy']

Running pipeline for Marcus...
  Done. Overall level: beginner, priority order: ['online_safety', 'responsible_ai_use', 'media_literacy', 'file_device_management']

All personas complete.


In [7]:
# Scoring based on each of the four criteria

for name, r in results.items():
    print("=" * 60)
    print(f"  PERSONA: {name.upper()}")
    print(f"  inferred_comfort: {r['profile']['inferred_comfort']}")
    print(f"  overall_level:    {r['overall_level']}")
    print(f"  priority_order:   {r['prioritized_areas']}")
    print()

    print("  QUIZ QUESTIONS:")
    for q in r['quiz_data']['questions']:
        print(f"    [{q['area']}] {q['question']}")
        for opt in q['options']:
            marker = " <-- correct" if opt.startswith(q['correct_answer']) else ""
            print(f"      {opt}{marker}")
    print()

    print("  SKILL MATRIX:")
    for area, data in r['skill_matrix'].items():
        print(f"    {area}: {data['score']} — {data['note']}")
    print()

    print("  LESSON HOOKS:")
    for lesson in r['curriculum']:
        print(f"    [{lesson['area']}] {lesson['title']}")
        print(f"      Hook: {lesson['interest_hook']}")
        print(f"      Activity preview: {lesson['activity'][:120]}...")
    print()



  PERSONA: JORDAN
  inferred_comfort: beginner
  overall_level:    beginner
  priority_order:   ['online_safety', 'responsible_ai_use', 'media_literacy', 'file_device_management']

  QUIZ QUESTIONS:
    [online_safety] You are playing your favorite online multiplayer game, and a new teammate asks for your real name and the name of your school so you can 'be better friends.' What is the safest thing to do?
      A. Tell them your real name but not your school.
      B. Give a fake school name but your real first name.
      C. Ignore the question or say you don't share personal info online. <-- correct
      D. Tell them the information because you are on the same team.
    [online_safety] You are reading the comments on a YouTube video about drawing setups. A comment says, 'Click here to get a free drawing tablet!' and includes a link. What should you do?
      A. Click the link to see if the offer is real.
      B. Do not click the link, because it could be a scam or a virus. <-- corr

# **Step 4:** Filling in/printing the scorecard

In [8]:
# Filling in the scores using the output from the above cell

scorecard = {
    "Jordan": {
        "quiz_difficulty_match":     {"score": "pass", "note": ""},
        "interest_integration_quiz": {"score": "pass", "note": ""},
        "lesson_hook_specificity":   {"score": "pass", "note": ""},
        "lesson_difficulty_match":   {"score": "pass", "note": ""},
        "priority_order_logic":      {"score": "pass", "note": ""}
    },
    "Sarah": {
        "quiz_difficulty_match":     {"score": "pass", "note": "This isn't a fail, but Sarah got everything right, which might mean the quiz wasn't hard enough to find her weak spots, or that she's genuinely strong across all areas. It's hard to tell with simulated answers."},
        "interest_integration_quiz": {"score": "pass", "note": ""},
        "lesson_hook_specificity":   {"score": "pass", "note": ""},
        "lesson_difficulty_match":   {"score": "pass", "note": ""},
        "priority_order_logic":      {"score": "partial", "note": "Sarah scored 'strong' in all four areas, so there wasn't really a weakest area to put first. The pipeline picked responsible_ai_use first, but the scores were all the same so it didn't really matter which one went first. Priority ordering works correctly when there's score variance it isn't really meaningful here."}
    },
    "Marcus": {
        "quiz_difficulty_match":     {"score": "pass", "note": ""},
        "interest_integration_quiz": {"score": "pass", "note": ""},
        "lesson_hook_specificity":   {"score": "pass", "note": ""},
        "lesson_difficulty_match":   {"score": "pass", "note": ""},
        "priority_order_logic":      {"score": "pass", "note": "Marcus and Jordan ended up with the exact same priority order, which looks a little suspicious. It's because the fixed random seed (random.seed(42)) gives both beginner personas the same answer pattern, not because the pipeline actually reasoned about Marcus specifically."}
    }
}

print("Scorecard ready to fill in. Run the summary cell after scoring.")

Scorecard ready to fill in. Run the summary cell after scoring.


In [9]:
# Printing the results for each persona

SCORE_SYMBOL = {"pass": "PASS", "partial": "PART", "fail": "FAIL", "": "----"}

criteria = [
    "quiz_difficulty_match",
    "interest_integration_quiz",
    "lesson_hook_specificity",
    "lesson_difficulty_match",
    "priority_order_logic"
]

names = list(scorecard.keys())
col_w = 10

print(f"{'Criterion':<30}" + "".join(f"{n:>{col_w}}" for n in names))
print("-" * (30 + col_w * len(names)))

for c in criteria:
    row = f"{c:<30}"
    for n in names:
        symbol = SCORE_SYMBOL.get(scorecard[n][c]["score"], "?")
        row += f"{symbol:>{col_w}}"
    print(row)

print()
print("NOTES:")
for n in names:
    print(f"\n  {n}:")
    for c in criteria:
        note = scorecard[n][c]['note']
        if note:
            print(f"    {c}: {note}")

Criterion                         Jordan     Sarah    Marcus
------------------------------------------------------------
quiz_difficulty_match               PASS      PASS      PASS
interest_integration_quiz           PASS      PASS      PASS
lesson_hook_specificity             PASS      PASS      PASS
lesson_difficulty_match             PASS      PASS      PASS
priority_order_logic                PASS      PART      PASS

NOTES:

  Jordan:

  Sarah:
    quiz_difficulty_match: This isn't a fail, but Sarah got everything right, which might mean the quiz wasn't hard enough to find her weak spots, or that she's genuinely strong across all areas. It's hard to tell with simulated answers.
    priority_order_logic: Sarah scored 'strong' in all four areas, so there wasn't really a weakest area to put first. The pipeline picked responsible_ai_use first, but the scores were all the same so it didn't really matter which one went first. Priority ordering works correctly when there's score varian